# skills

> Instructions an agent reads when it needs them, and extensions a user can drop in.

A skill is a document that tells an agent how to do one kind of task: how to use a library, the house style of a project, the steps of a release. Putting every skill into the system prompt would waste context, so the prompt lists only each skill's name and a one-line description, and the agent reads a skill in full with the `read_skill` tool when a task calls for it. This module finds skills, renders that list, and looks a skill up by name.

It also loads extensions: Python files a user drops in a folder to add tools, skills, slash commands, lifecycle hooks and an approval policy to an agent.

In [ ]:
#| default_exp skills

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import fcntl, fnmatch, inspect, json, runpy, subprocess, threading, tomllib
from contextvars import ContextVar, copy_context
from functools import update_wrapper, wraps
from importlib.metadata import entry_points
from fastcore.basics import AttrDict
from fastcore.parallel import startthread
from fastcore.xtras import atomic_save
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.docments import frontmatter
from shalya.host import host_err

## A skill

Skills come from two sources. A pyskill is a Python module whose docstring is the skill, published by an installed package (see [pyskills](https://github.com/AnswerDotAI/pyskills)). A `SKILL.md` file is a markdown file in a skills folder, in the Agent Skills layout. Both become a `Skill`, with a name, a source, a description, and `where` it came from so a person can go and read it.

A `Skill` holds a loader for its body, not the body, so finding skills does not read every file. When no description is given, it is the body's first paragraph on one line.

In [ ]:
#| export
GROUP, EXTRA_MODULES, MAX_SKILL_CHARS = 'pyskills', ('exhash.skill',), 20_000

`GROUP` is the entry-point group installed packages publish pyskills under. `EXTRA_MODULES` lists modules that are skills but do not publish one. `MAX_SKILL_CHARS` caps the length of a skill body sent to the model.

In [ ]:
GROUP, EXTRA_MODULES, MAX_SKILL_CHARS

('pyskills', ('exhash.skill',), 20000)

In [ ]:
test_eq(GROUP, 'pyskills')
test_eq(EXTRA_MODULES, ('exhash.skill',))
test_eq(MAX_SKILL_CHARS, 20_000)

In [ ]:
#| export
def describe(text, mx=300):
    'A one-line description from a skill body: its first paragraph, collapsed.'
    body = (text or '').strip()
    if not body: return ''
    para = body.split('\n\n', 1)[0]
    one = ' '.join(para.split())
    return one if len(one) <= mx else one[:mx - 1].rstrip() + '…'

_describe = describe   # the old name
@dataclass
class Skill:
    'One skill: its name, where it came from, when it applies, and a way to load its full text.'
    name: str
    source: str                   # 'pyskill' | 'md'
    description: str = ''
    where: str = ''               # module path or file path, shown so a person can go read it
    _text: object = field(default=None, repr=False)

    def text(self):
        'The full skill body, clipped to `MAX_SKILL_CHARS`; a skill that cannot be read returns a message saying so.'
        try: t = self._text() if callable(self._text) else (self._text or '')
        except Exception as e: return f'could not read skill {self.name}: {host_err(e)}'
        t = str(t)
        return t if len(t) <= MAX_SKILL_CHARS else t[:MAX_SKILL_CHARS] + f'\n…[{len(t)-MAX_SKILL_CHARS} more chars]'

    def dict(self): return {'name': self.name, 'source': self.source,'description': self.description, 'where': self.where}

In [ ]:
describe('''Search the web and read results into durable memory.

Use it when a question needs pages nobody has read yet.''')

'Search the web and read results into durable memory.'

In [ ]:
test_eq(describe(''), '')
test_eq(describe('one\ntwo\n\nthree'), 'one two')
test_eq(len(describe('x ' * 400)), 300)

`text()` loads the body when it is asked for and clips it to `MAX_SKILL_CHARS`. A skill that cannot be read returns a message saying so, instead of raising an exception into the agent's turn.

In [ ]:
s = Skill('nbdev', 'md', 'Develop nbdev projects.', 'nbs/SKILL.md', _text=lambda: 'the whole body')
def explodes(): raise FileNotFoundError('SKILL.md')
broken = Skill('gone', 'md', _text=explodes)
s.text(), broken.text()

('the whole body', 'could not read skill gone: FileNotFoundError: SKILL.md')

In [ ]:
test_eq(s.text(), 'the whole body')
assert broken.text().startswith('could not read skill gone'), broken.text()
big = Skill('big', 'md', _text='x' * (MAX_SKILL_CHARS + 500))
test_eq(len(big.text()), MAX_SKILL_CHARS + len('\n…[500 more chars]'))
test_eq(s.dict()['where'], 'nbs/SKILL.md')

## Where skills come from

Installed packages publish pyskills under the `pyskills` entry-point group. A module's description comes from its docstring, and Python offers no way to read a docstring without importing the module, so discovery imports each pyskill module once. A module with an empty docstring is skipped.

In [ ]:
#| export
def mod_skill(name, modpath):
    "A `Skill` for a module, without importing it until someone asks for the body."
    def load():
        from importlib import import_module
        return import_module(modpath).__doc__ or ''
    # the description needs the docstring, and there is no way to read one without importing
    try:
        from importlib import import_module
        doc = import_module(modpath).__doc__ or ''
    except Exception: return None
    if not doc.strip(): return None
    return Skill(name=name, source='pyskill', description=describe(doc), where=modpath, _text=load)

_mod_skill = mod_skill   # the old name
def _pyskills():
    "Every module published under the `pyskills` entry-point group, plus the known stragglers."
    out, seen = [], set()
    try:
        from importlib.metadata import entry_points
        eps = list(entry_points(group=GROUP))
    except Exception:
        eps = []
    for ep in eps:
        mod = getattr(ep, 'value', None) or ep.name
        if mod in seen: continue
        seen.add(mod)
        if (s := mod_skill(ep.name.split('.')[-1] or ep.name, mod)): out.append(s)
    for mod in EXTRA_MODULES:
        if mod in seen: continue
        seen.add(mod)
        if (s := mod_skill(mod.split('.')[0], mod)): out.append(s)
    return out

In [ ]:
assert _mod_skill is mod_skill   # the old name still imports
assert _describe is describe

`exhash` ships a skill module and is a dependency of shalya, so discovery always finds at least one real pyskill.

In [ ]:
found = _pyskills()
[(s.name, s.source, s.where) for s in found]

[('editskill', 'pyskill', 'fastcore.editskill'),
 ('rgapi', 'pyskill', 'rgapi.skill'),
 ('ghapi', 'pyskill', 'ghapi.skill'),
 ('kosha', 'pyskill', 'kosha.skill'),
 ('exhash', 'pyskill', 'exhash.skill'),
 ('skill', 'pyskill', 'nbdev.skill'),
 ('skill', 'pyskill', 'pyskills.skill'),
 ('skill', 'pyskill', 'llmsurgery.skill'),
 ('fastcdp', 'pyskill', 'fastcdp.skill'),
 ('dlgskill', 'pyskill', 'aidialog.dlgskill'),
 ('vishalakshi', 'pyskill', 'vishalakshi.skill'),
 ('coding_patterns', 'pyskill', 'ramabana.coding_patterns'),
 ('theory', 'pyskill', 'ramabana.theory'),
 ('write_docs', 'pyskill', 'ramabana.write_docs'),
 ('write_prose', 'pyskill', 'ramabana.write_prose')]

In [ ]:
assert any(s.where == 'exhash.skill' for s in found), [s.where for s in found]
ex = next(s for s in found if s.where == 'exhash.skill')
assert ex.description, 'a pyskill with no description should have been dropped'
assert len(ex.text()) > 100, len(ex.text())

`SKILL.md` files are looked for in a list of folders, from lowest to highest precedence, so that a later folder's skill replaces an earlier one with the same name. `skill_dirs` returns the list. It starts with the `skills` folder inside `learned`, for skills a tool wrote rather than a person, then the application's config folder `cfg`, then `~/.agents/skills` for a person's own skills, then `.leela/skills` and `.agents/skills` in each project root. A project can therefore replace what a person set up for themselves, and a skill a tool learned replaces an installed pyskill but loses to every folder a person writes.

The `read_skill` tool reads the same installed skill that discovery found through the `pyskills` entry point:

In [ ]:
from shalya.host import LocalHost
from shalya.skills import discover
from shalya.tools import skill_tools

In [ ]:
installed_skills = discover()
installed_host = LocalHost(index=False)
installed_read = {t.__name__: t for t in skill_tools(installed_host, lambda: installed_skills)}['read_skill']
installed_exhash = next(s for s in installed_skills if s.where == 'exhash.skill')
installed_body = installed_read(installed_exhash.name)
assert installed_body.startswith(f'<skill name="{installed_exhash.name}"')
assert len(installed_body) > 100 and 'exhash' in installed_body.lower()

In [ ]:
#| export
def skill_dirs(roots=(), cfg=None, learned=None):
    "Where SKILL.md files are looked for, in increasing precedence: `learned` first, then the user's, then each project's."
    from pathlib import Path
    ds = []
    if learned is not None: ds.append(Path(learned)/'skills')
    if cfg is not None: ds.append(Path(cfg)/'skills')
    ds.append(Path.home()/'.agents'/'skills')
    for r in roots: ds += [Path(r)/'.leela'/'skills', Path(r)/'.agents'/'skills']
    return ds

def _md_skills(d):
    "Skills in one directory, following the Agent Skills layout: `<name>/SKILL.md`."
    from pathlib import Path
    d = Path(d)
    if not d.is_dir(): return []
    out = []
    for p in sorted(d.iterdir()):
        if not p.is_dir(): continue
        f = p/'SKILL.md'
        if not f.exists(): continue
        try: raw = f.read_text(encoding='utf-8')
        except Exception: continue
        meta, body = frontmatter(raw)
        out.append(Skill(name=meta.get('name') or p.name, source='md',
                         description=meta.get('description') or describe(body),
                         where=str(f), _text=body))
    return out

def discover(roots=(), cfg=None, extra=(), learned=None):
    "Every skill available to the agent, sorted by name: pyskills, then `skill_dirs`, then `extra`, a later one replacing an earlier one of the same name."
    by_name = {}
    for s in _pyskills(): by_name[s.name] = s
    for d in skill_dirs(roots, cfg, learned):
        for s in _md_skills(d): by_name[s.name] = s
    for s in extra or (): by_name[s.name] = s
    return sorted(by_name.values(), key=lambda s: s.name)

In [ ]:
[str(p) for p in skill_dirs(roots=['/proj'], cfg='/home/k/.config/leela')]

['/home/k/.config/leela/skills',
 '/Users/71293/.agents/skills',
 '/proj/.leela/skills',
 '/proj/.agents/skills']

In [ ]:
ds = [str(p) for p in skill_dirs(roots=['/proj'], cfg='/cfg')]
test_eq(ds[0], '/cfg/skills')
test_eq(ds[-1], '/proj/.agents/skills')
test_eq(str(skill_dirs(roots=['/proj'], cfg='/cfg', learned='/cfg/learned')[0]), '/cfg/learned/skills')

A skills folder follows the Agent Skills layout: one folder per skill holding a `SKILL.md`, whose optional frontmatter gives the name and description. Without frontmatter the folder name is the skill's name and the first paragraph its description.

In [ ]:
tmp = Path(tempfile.mkdtemp())
d = tmp/'.agents'/'skills'/'notebook-tests'
d.mkdir(parents=True)
(d/'SKILL.md').write_text('''---
name: notebook-tests
description: Write executable test cells in the notebook that owns the code.
---

Put the readable case in the notebook and the bulk assertions in `tests/`.
''')
plain = tmp/'.agents'/'skills'/'plain'
plain.mkdir()
(plain/'SKILL.md').write_text('Say what changed.\n\nAnd nothing else.\n')
[(s.name, s.description) for s in _md_skills(tmp/'.agents'/'skills')]

[('notebook-tests',
  'Write executable test cells in the notebook that owns the code.'),
 ('plain', 'Say what changed.')]

In [ ]:
got = {s.name: s for s in _md_skills(tmp/'.agents'/'skills')}
test_eq(sorted(got), ['notebook-tests', 'plain'])
test_eq(got['plain'].description, 'Say what changed.')     # no frontmatter: the first paragraph
assert 'bulk assertions' in got['notebook-tests'].text()
test_eq(got['notebook-tests'].source, 'md')

`discover` collects every source: pyskills, then the `skill_dirs` folders, then the `extra` skills you pass. A later source replaces an earlier skill of the same name, and the result is sorted by name.

In [ ]:
skills = discover(roots=[tmp])
[s.name for s in skills]

['coding_patterns',
 'design-taste-frontend',
 'dlgskill',
 'editskill',
 'exhash',
 'fastcdp',
 'find-skills',
 'full-output-enforcement',
 'ghapi',
 'hf-cli',
 'high-end-visual-design',
 'industrial-brutalist-ui',
 'kosha',
 'minimalist-ui',
 'nbdev',
 'notebook-tests',
 'plain',
 'redesign-existing-projects',
 'rgapi',
 'skill',
 'stitch-design-taste',
 'theory',
 'vishalakshi',
 'write_docs',
 'write_prose']

In [ ]:
names = [s.name for s in skills]
assert 'notebook-tests' in names and 'exhash' in names, names
test_eq(names, sorted(names))
mine = discover(roots=[tmp], extra=[Skill('plain', 'ext', 'Mine wins.')])
test_eq(next(s for s in mine if s.name == 'plain').description, 'Mine wins.')
learned = tmp/'learned'
for name, desc in [('notebook-tests', 'Learned version.'), ('triage', 'Only learned.')]:
    (learned/'skills'/name).mkdir(parents=True)
    (learned/'skills'/name/'SKILL.md').write_text(f'---\nname: {name}\ndescription: {desc}\n---\n\nbody\n')
got = {s.name: s.description for s in discover(roots=[tmp], learned=learned)}
test_eq(got['triage'], 'Only learned.')
test_eq(got['notebook-tests'], 'Write executable test cells in the notebook that owns the code.')   # a person's folder outranks a learned one

## The index

`skill_index` renders the block that goes into the system prompt: a heading, a line telling the agent to call `read_skill(name)` before doing the work a skill covers, and one line per skill with its name and description, never its body. Each description is cut to `SKILL_DESC_MAX` characters at a word boundary, so one long description cannot crowd out the rest.

In [ ]:
#| export
SKILL_DESC_MAX = 160   # per skill. One verbose description cannot crowd out the rest

def _clip_desc(s, n=SKILL_DESC_MAX):
    "One line, clipped at a word boundary: in the index a description only has to be pickable."
    s = ' '.join(str(s).split())
    if len(s) <= n: return s
    cut = s.rfind(' ', 0, n)
    return s[:cut if cut > 0 else n].rstrip(' .,;:\u2014-') + '\u2026'

def skill_index(skills):
    "The system-prompt block listing each skill's name and clipped description."
    if not skills: return ''
    rows = '\n'.join(f'- `{s.name}`: {_clip_desc(s.description)}' for s in skills)
    return ('\n\n## Skills\n\nUse `read_skill(name)` before doing the work a skill covers. '
            'Installed tool code is also searchable with `search_code`.\n\n' + rows)

def find(skills, name):
    "A skill by exact name, then unique prefix, then unique substring. Ambiguity is None, not a guess."
    if not name: return None
    n = name.strip().lower()
    if (exact := [s for s in skills if s.name.lower() == n]): return exact[0]
    for pred in (lambda s: s.name.lower().startswith(n), lambda s: n in s.name.lower()):
        if len(hits := [s for s in skills if pred(s)]) == 1: return hits[0]
    return None

In [ ]:
print(skill_index([s for s in skills if s.name == 'notebook-tests']))



## Skills

Know-how available to you. Read one with `read_skill(name)` when its description matches what you are about to do, *before* you do it -- several of these describe tools already installed in this environment, so the code they discuss is also searchable with `search_code`.

- `notebook-tests` -- Write executable test cells in the notebook that owns the code.


In [ ]:
ix = skill_index([s for s in skills if s.name == 'notebook-tests'])
assert '`notebook-tests`' in ix and 'read_skill' in ix
assert 'Put the readable case' not in ix, 'the index must not carry a body'
test_eq(skill_index([]), '')
test_eq(len(_clip_desc('word ' * 60)), SKILL_DESC_MAX)

`find` looks a skill up by exact name, then by unique prefix, then by unique substring. A name that matches several skills returns None instead of guessing, and the `read_skill` tool then lists the skills that exist.

In [ ]:
find(skills, 'notebook-tests').name, find(skills, 'notebook').name, find(skills, 'nope')

('notebook-tests', 'notebook-tests', None)

In [ ]:
test_eq(find(skills, 'notebook-tests').name, 'notebook-tests')
test_eq(find(skills, 'notebook').name, 'notebook-tests')     # unique prefix
test_eq(find(skills, 'tests').name, 'notebook-tests')        # unique substring
test_eq(find(skills, 'nope'), None)
two = [Skill('edit', 'md'), Skill('editor', 'md')]
test_eq(find(two, 'edit').name, 'edit')                      # exact beats prefix
test_eq(find(two, 'edi'), None)                              # ambiguous prefix

## Extensions

An extension is a Python file a user drops in an extensions folder. It defines `setup(reg)`, and through `reg`, a `Registry`, it may add tools, skills and slash commands, hook lifecycle events, and replace the approval policy. That is all a `Registry` offers: it gives an extension no access to the model, the engine or the conversation history.

In [ ]:
#| export
EVENTS = ('session_start', 'before_turn', 'after_turn', 'before_tool', 'after_tool', 'stop', 'compact', 'approval', 'background_done', 'watch', 'session_end')

`EVENTS` lists the lifecycle events an extension may hook, such as `before_tool` and `after_turn`. Hooking any other name raises an error, so a typo cannot silently hook nothing.

In [ ]:
EVENTS

('before_turn',
 'after_turn',
 'before_tool',
 'after_tool',
 'compact',
 'approval')

In [ ]:
test_eq(EVENTS, ('session_start', 'before_turn', 'after_turn', 'before_tool', 'after_tool', 'stop', 'compact', 'approval', 'background_done', 'watch', 'session_end'))
test_eq(len(set(EVENTS)), len(EVENTS))

In [ ]:
#| export
class _Callback:
    def __init__(self, reg, fn, kind, owner, rows=None):
        self.reg, self.fn, self.kind, self.owner = reg, fn, kind, owner
        self.rows, self.failed = rows, False
        self.errors = set()
        self.lock = threading.RLock()
        update_wrapper(self, fn, updated=())

    def __call__(self, *args, **kw):
        with self.lock:
            if self.failed or (self.owner is not None and not self.owner.on): return None
        token = self.reg._context.set((self.owner, False))
        try: return self.fn(*args, **kw)
        except Exception as e:
            if self.owner is None and self.kind.endswith(' hook'): raise
            decision = self.kind in ('before_tool hook', 'before_turn hook', 'approval')
            error = host_err(e)
            with self.lock:
                if error not in self.errors:
                    self.errors.add(error)
                    self.reg._note(self.owner, f'{self.kind} failed: {error}')
                if not decision: self.failed = True
            if decision:
                return 'deny' if self.kind == 'approval' else f'mod {self.owner.name} failed: {e}'
        finally: self.reg._context.reset(token)

class _Timer:
    def __init__(self, seconds, callback, repeat):
        if seconds < 0 or (repeat and seconds == 0): raise ValueError('timer interval must be positive')
        self.seconds, self.callback, self.repeat = seconds, callback, repeat
        self.stop = threading.Event()
        self.thread = None

    @property
    def cancelled(self): return self.stop.is_set()

    def cancel(self):
        self.stop.set()
        self._remove()

    def _remove(self):
        reg = self.callback.reg
        with reg._lock:
            reg.timers[:] = [t for t in reg.timers if t is not self]
            reg._entries[:] = [e for e in reg._entries if e.value is not self]

    def start(self):
        self.thread = startthread(self.run, daemon=True)

    def run(self):
        try:
            while not self.stop.wait(self.seconds):
                self.callback()
                if not self.repeat or self.callback.failed: break
        finally: self._remove()

class Registry:
    "What an extension's `setup(reg)` may add: tools, skills, commands, hooks and an approval policy."

    def __init__(self, host=None, agent=None):
        self.host, self.agent = host, agent
        self.tools, self.skills, self.commands = [], [], {}
        self.hooks = {e: [] for e in EVENTS}
        self.approve = None
        self.notes, self.mods = [], []
        self.status_segments, self.bands, self.timers = [], [], []
        self.on_redraw = None
        self._base, self._dry = None, False
        self._entries, self._notices = [], []
        self._lock = threading.RLock()
        self._status_state = threading.local()
        self._context = ContextVar('mod_context', default=(None, False))


    def tool(self, f):
        "Register a tool."
        self._add('tool', self._owned(f))
        return f

    def skill(self, name, text, description=''):
        "Register a skill."
        s = Skill(name=name, source='ext', description=description or describe(text if isinstance(text, str) else ''),
                  where='extension', _text=text)
        self._add('skill', s)
        return s

    def command(self, name, fn, help=''):
        "Add a slash command. `fn(agent, arg)` returns text for the frontend to show."
        self._add('command', (self._owned(fn), help), name.lstrip('/'))
        return fn

    def on(self, event, fn):
        "Register a hook for a known lifecycle event."
        if event not in EVENTS: raise KeyError(f'unknown event {event!r}; known: {", ".join(EVENTS)}')
        self._add('hook', self._callback(fn, f'{event} hook'), event)
        return fn

    def approval(self, fn):
        "Replace the approval policy; the last registration wins."
        self._add('approval', self._callback(fn, 'approval') if self.mod is not None else fn)
        return fn


    @property
    def loaded(self):
        "Whether the extension files have been read into this registry."
        return self._base is not None

    def mark(self):
        "What is registered right now. `load` brackets itself with two of these."
        class Mark(tuple): pass
        mark = Mark((len(self.tools), len(self.skills), len(self.notes), dict(self.commands),
                     {e: len(v) for e, v in self.hooks.items()}, self.approve))
        mark._entries, mark._mods = list(self._entries), list(self.mods)
        return mark

    def mark_loaded(self, before):
        "Record the span the extension files registered inside, from the `mark` taken before them."
        prior = getattr(before, '_entries', None)
        if prior is None:
            counts = dict(tool=before[0], skill=before[1])
            hooks = dict(before[4])
            prior = []
            for e in self._entries:
                if e.kind in counts and counts[e.kind]:
                    prior.append(e)
                    counts[e.kind] -= 1
                elif e.kind == 'hook' and hooks.get(e.key, 0):
                    prior.append(e)
                    hooks[e.key] -= 1
                elif e.kind == 'command' and before[3].get(e.key) is e.value: prior.append(e)
                elif e.kind == 'approval' and before[5] is e.value: prior.append(e)
        self._base = (before, self.mark(), [e for e in self._entries if all(e is not old for old in prior)],
                      [m for m in self.mods if all(m is not old for old in getattr(before, '_mods', ()))])
        return self

    def drop_loaded(self):
        "Forget what the extension files registered, keeping what was registered outside that span."
        if self._base is None: return self
        before, after, entries, mods = self._base
        for mod in mods: self.off(mod.name)
        self._remove(entries)
        with self._lock:
            self.mods = [m for m in self.mods if all(m is not old for old in mods)]
            self.notes = self.notes[:before[2]] + self.notes[after[2]:]
            self._base = None
        return self

    def fire(self, event, *args, **kw):
        "Run hooks for `event`, recording failures. Returns what each surviving hook returned, in registration order."
        out = []
        for f in list(self.hooks.get(event, ())):
            if isinstance(f, _Callback) and (f.failed or (f.owner is not None and not f.owner.on)): continue
            try:
                result = f(*args, **kw)
                if not isinstance(f, _Callback) or not f.failed: out.append(result)
            except Exception as e:
                with self._lock: self.notes.append(f'{event} hook failed: {host_err(e)}')
        return out

    def each(self, event):
        "Return live hooks with their owners in registration order."
        with self._lock: hooks = list(self.hooks.get(event, ()))
        out = []
        for fn in hooks:
            callback = fn if isinstance(fn, _Callback) else _Callback(self, fn, f'{event} hook', None)
            if callback.failed or (callback.owner is not None and not callback.owner.on): continue
            @wraps(callback)
            def call(*args, _callback=callback, **kw):
                try: return _callback(*args, **kw)
                except Exception as e:
                    with self._lock: self.notes.append(f'{event} hook failed: {host_err(e)}')
            out.append((callback.owner.name if callback.owner is not None else None, call))
        return out

    @property
    def mod(self):
        "Return the name of the calling mod."
        owner, setup = self._context.get()
        return owner.name if owner is not None else None

    @property
    def home(self):
        "Return the home folder of the calling mod."
        owner, setup = self._context.get()
        return owner.home if owner is not None else None

    def _owned(self, fn):
        owner = self._context.get()[0]
        if owner is None: return fn
        @wraps(fn)
        def call(*args, **kw):
            token = self._context.set((owner, False))
            try: return fn(*args, **kw)
            finally: self._context.reset(token)
        return call

    def thread(self, fn, *args, **kw):
        "Start a daemon thread with the calling mod's context."
        context = copy_context()
        context.run(self._context.set, (self._context.get()[0], False))
        thread = threading.Thread(target=lambda: context.run(fn, *args, **kw), daemon=True)
        if not self._dry: thread.start()
        return thread

    def store(self, default=None):
        "Return the calling mod's persistent JSON store."
        owner = self._context.get()[0]
        if owner is None: raise RuntimeError('store requires a mod context')
        with self._lock:
            if getattr(owner, '_store', None) is None: owner._store = _Store(owner.home, default)
            return owner._store

    def _add(self, kind, value, key=None):
        owner = self._context.get()[0]
        with self._lock:
            if owner is not None and not owner.on: return value
            entry = AttrDict(owner=owner, kind=kind, value=value, key=key, previous=None)
            if kind == 'command':
                entry.previous = self.commands.get(key)
                self.commands[key] = value
            elif kind == 'approval':
                entry.previous, self.approve = self.approve, value
            else:
                target = self.hooks[key] if kind == 'hook' else getattr(self, {'tool':'tools', 'skill':'skills', 'status':'status_segments', 'band':'bands', 'timer':'timers'}[kind])
                target.append(value)
            self._entries.append(entry)
        return value

    def _remove(self, entries):
        with self._lock:
            for entry in reversed(entries):
                if not any(e is entry for e in self._entries): continue
                kind, value, key = entry.kind, entry.value, entry.key
                if kind in ('command', 'approval'):
                    for later in self._entries:
                        if later.kind == kind and later.key == key and later.previous is value:
                            later.previous = entry.previous
                    if kind == 'command' and self.commands.get(key) is value:
                        if entry.previous is None: self.commands.pop(key, None)
                        else: self.commands[key] = entry.previous
                    elif kind == 'approval' and self.approve is value: self.approve = entry.previous
                else:
                    target = self.hooks[key] if kind == 'hook' else getattr(self, {'tool':'tools', 'skill':'skills', 'status':'status_segments', 'band':'bands', 'timer':'timers'}[kind])
                    for i, item in enumerate(target):
                        if item is value:
                            target.pop(i)
                            break
                    if kind == 'timer': value.cancel()
                self._entries = [e for e in self._entries if e is not entry]

    def _callback(self, fn, kind):
        return _Callback(self, fn, kind, self._context.get()[0])

    def _note(self, owner, text):
        with self._lock:
            if owner is not None: owner.note = f'{owner.note}; {text}' if owner.note else text
            label = owner.path.name if owner is not None and owner.path is not None else owner.name if owner is not None else None
            self.notes.append(f'{label}: {text}' if label is not None else text)

    def status(self, fn):
        "Add a status segment."
        self._add('status', self._callback(fn, 'status'))
        return fn

    def status_texts(self, agent):
        "Return live nonempty status strings without same-thread re-entry."
        if getattr(self._status_state, 'active', False): return []
        self._status_state.active = True
        try:
            with self._lock: segments = list(self.status_segments)
            return [text for fn in segments if isinstance(text := fn(agent), str) and text]
        finally: self._status_state.active = False

    def band(self, fn, rows=6):
        "Add a band above the prompt."
        if type(rows) is not int or not 1 <= rows <= 50: raise ValueError('rows must be an integer from 1 to 50')
        callback = self._callback(fn, 'band')
        callback.rows = rows
        self._add('band', callback)
        return fn

    def notify(self, text):
        "Queue a notice from any thread."
        owner = self._context.get()[0]
        with self._lock: self._notices.append(dict(mod=owner.name if owner is not None else None, text=text))
        self.redraw()

    def drain_notices(self):
        "Return and clear queued notices in order."
        with self._lock:
            notices, self._notices = self._notices, []
        return notices

    def redraw(self):
        "Ask the frontend to draw again."
        if self.on_redraw is not None: self.on_redraw()

    def _timer(self, seconds, fn, repeat):
        timer = _Timer(seconds, self._callback(fn, 'timer'), repeat)
        self._add('timer', timer)
        owner, setup = self._context.get()
        if not self._dry and not setup:
            with self._lock:
                if owner is None or owner.on: timer.start()
                else: timer.cancel()
        return timer

    def after(self, seconds, fn):
        "Run a callback once after a delay."
        return self._timer(seconds, fn, False)

    def every(self, seconds, fn):
        "Run a callback repeatedly until cancelled."
        return self._timer(seconds, fn, True)

    def off(self, name):
        "Remove one mod's registrations and cancel its timers."
        mod = self._find_mod(name)
        with self._lock:
            mod.on = False
            self._remove([e for e in self._entries if e.owner is mod])
        return self

    def on_mod(self, name):
        "Run a disabled mod's setup again."
        mod = self._find_mod(name)
        with self._lock:
            if mod.on: return self
            mod.on, mod.note = True, ''
        token = self._context.set((mod, True))
        try:
            if mod.setup is None:
                if mod.path is None: mod.setup = mod.entry.load()
                else:
                    if mod.path.is_dir() and (mod.path/'mod.toml').exists():
                        meta = tomllib.loads((mod.path/'mod.toml').read_text())
                        mod.description, mod.version = meta.get('description', ''), meta.get('version', '')
                    target = mod.path/'mod.py' if mod.path.is_dir() else mod.path
                    mod.setup = runpy.run_path(str(target)).get('setup')
            if not callable(mod.setup):
                self._note(mod, 'loaded, no setup()')
            else:
                mod.setup(self)
            if mod.path is not None and mod.path.is_dir():
                for skill in _md_skills(mod.path/'skills'):
                    if not skill.name.startswith(('_', '.')): self._add('skill', skill)
            entries = [e for e in self._entries if e.owner is mod]
            counts = {k: sum(e.kind == k for e in entries) for k in ('tool', 'skill', 'command', 'hook', 'approval', 'status', 'band', 'timer')}
            if callable(mod.setup):
                summary = ', '.join(f'{n} {k if n == 1 else "statuses" if k == "status" else k + "s"}' for k, n in counts.items() if n)
                self._note(mod, summary or 'nothing registered')
            if not self._dry:
                for e in entries:
                    if e.kind == 'timer':
                        with self._lock:
                            if mod.on and not e.value.cancelled: e.value.start()
        except (Exception, SystemExit) as e:
            phase = 'setup() failed' if callable(mod.setup) else 'failed to load'
            self._note(mod, f'{phase} ({host_err(e)})')
            self.off(name)
        finally: self._context.reset(token)
        return self

    def _find_mod(self, name):
        mod = next((m for m in self.mods if m.name == name), None)
        if mod is None: raise KeyError(f'unknown mod {name!r}; known: {", ".join(m.name for m in self.mods)}')
        return mod

    def add_mod(self, name, setup, home=None):
        "Register a setup callable under its mod context and return the mod record."
        return self._mod(name, 'callable', setup=setup, home=home)

    def _mod(self, name, source, path=None, entry=None, cfg=None, only=None, setup=None, home=None):
        if setup is None and name.startswith(('_', '.')): return
        with self._lock:
            old = next((m for m in self.mods if m.name == name), None)
            if old is not None:
                self.off(name)
                self.mods.remove(old)
        if home is not None: home = Path(home)
        elif path is not None:
            if path.is_dir(): home = path
            elif path.parent.name == 'extensions': home = path.parent.parent/'mods'/name
            elif path.parent.name == 'mods' or cfg is None: home = path.parent/name
            else: home = Path(cfg)/'mods'/name
        elif cfg is not None: home = Path(cfg)/'mods'/name
        mod = AttrDict(name=name, source=source, on=False, note='', home=home, path=path, entry=entry,
                       setup=setup, description='', version='')
        with self._lock: self.mods.append(mod)
        if only is None or name in only: self.on_mod(name)
        else: self._note(mod, 'not in this vesha')
        return mod

def ext_dirs(roots=(), cfg=None, project=False):
    "The extension folders to load: `cfg/extensions`, and with `project=True` each root's `.leela/extensions`."
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'extensions')
    if project:
        for r in roots: ds.append(Path(r)/'.leela'/'extensions')
    return ds

def load(reg, roots=(), cfg=None, project=False, paths=(), only=None):
    "Run each extension file and call its `setup(reg)`, recording in `reg.notes` what each added or why it failed."
    mark = reg.mark()
    sources = {}
    for ep in entry_points(group='ramabana.mods'):
        if not ep.name.startswith(('_', '.')): sources[ep.name] = ('entry point', None, ep)
    bases = ([Path(cfg)] if cfg is not None else []) + ([Path(r)/'.leela' for r in roots] if project else [])
    for base in bases:
        for directory in (base/'extensions', base/'mods'):
            for p in _mod_paths(directory):
                name = p.stem if p.is_file() else p.name
                sources.pop(name, None)
                sources[name] = (str(p), p, None)
    for path in paths or ():
        p = Path(path)
        for item in ([p] if p.is_file() or (p/'mod.py').is_file() else _mod_paths(p)):
            if not item.name.startswith(('_', '.')):
                name = item.stem if item.is_file() else item.name
                sources.pop(name, None)
                sources[name] = (str(item), item, None)
    try:
        for name, (source, path, ep) in sources.items(): reg._mod(name, source, path, ep, cfg, only)
        if cfg is not None: shell_hooks(reg, Path(cfg)/'hooks.json')
    finally: reg.mark_loaded(mark)
    return reg

def _mod_paths(directory):
    "Return visible mod files followed by mod folders."
    directory = Path(directory)
    if not directory.is_dir(): return []
    visible = [p for p in sorted(directory.iterdir()) if not p.name.startswith(('_', '.'))]
    return [p for p in visible if p.is_file() and p.suffix == '.py'] + [p for p in visible if p.is_dir() and (p/'mod.py').is_file()]

def check(path):
    "Inspect one mod without starting registry timers."
    path = Path(path)
    reg = Registry()
    reg._dry = True
    reg._mod(path.stem if path.is_file() else path.name, str(path), path)
    return dict(tools=[getattr(f, '__name__', type(f).__name__) for f in reg.tools],
                tool_docs=[dict(name=getattr(f, '__name__', type(f).__name__), signature=str(inspect.signature(f)), doc=inspect.getdoc(f) or '') for f in reg.tools],
                commands=list(reg.commands), skills=[s.name for s in reg.skills],
                hooks={e: len(h) for e, h in reg.hooks.items() if h}, status=[getattr(f.fn, '__name__', type(f.fn).__name__) for f in reg.status_segments],
                bands=[dict(rows=b.rows) for b in reg.bands],
                timers=[dict(seconds=t.seconds, repeat=t.repeat) for t in reg.timers],
                mods=[dict(name=m.name, source=m.source, on=m.on, note=m.note) for m in reg.mods])

In [ ]:
#| export
HOOK_TIMEOUT = 30

def _hook_arg(a):
    "What a shell hook is told about one positional: an id where the object has one, else its text."
    return getattr(a, 'session_id', None) or getattr(a, 'id', None) or (a if isinstance(a, (str, int, float, dict, list, type(None))) else type(a).__name__)

def _shell_hook(event, command, match=''):
    "A hook that feeds `command` `{event, args, **kw}` as JSON and reads a JSON object back: `deny`, `args` or `result`; any other output is ignored."
    def hook(*args, **kw):
        name = kw.get('name') or (args[1] if len(args) > 1 else '')
        if match and not fnmatch.fnmatch(str(name), match): return None
        r = subprocess.run(command, shell=True, input=json.dumps(dict(event=event, args=[_hook_arg(a) for a in args], **kw), default=str),
                           capture_output=True, text=True, timeout=HOOK_TIMEOUT)
        out = r.stdout.strip()
        if not out.startswith('{'): return None
        d = json.loads(out)
        return d.get('deny') or d.get('args') or d.get('result')
    return hook

def shell_hooks(reg, path):
    "Register the hooks a `hooks.json` lists: `[{event, match, command}]`."
    path = Path(path)
    if not path.exists(): return reg
    try: entries = json.loads(path.read_text())
    except Exception as e:
        reg.notes.append(f'{path.name}: failed to load ({host_err(e)})')
        return reg
    n = 0
    for h in entries:
        ev, cmd = h.get('event'), h.get('command')
        if ev not in EVENTS or not cmd: reg.notes.append(f'{path.name}: skipped {h}'); continue
        reg.on(ev, _shell_hook(ev, cmd, h.get('match', ''))); n += 1
    reg.notes.append(f'{path.name}: {n} hook(s)')
    return reg

In [ ]:
#| export
class _Store:
    def __init__(self, home, default):
        self.path = Path(home)/'state.json' if home is not None else None
        self.default = dict(default or {})
        self._data = None
        self.problem = ''
        self._lock = threading.RLock()

    def _read(self):
        self.problem = ''
        if self.path is not None:
            try:
                data = json.loads(self.path.read_text())
                if not isinstance(data, dict): raise ValueError('expected a JSON object')
                return data
            except FileNotFoundError: pass
            except (OSError, ValueError) as e: self.problem = str(e).splitlines()[0][:160]
        return dict(self.default)

    @property
    def data(self):
        "Load the store's data on first access."
        with self._lock:
            if self._data is None: self._data = self._read()
            return self._data

    def reload(self):
        "Re-read the file and return its data or the default."
        with self._lock:
            self._data = self._read()
            return self._data

    def save(self):
        "Write data atomically unless the existing file cannot be read."
        with self._lock:
            data = self.data
            if self.problem: raise ValueError(f'{self.path}: {self.problem}')
            if self.path is not None:
                self._read()
                if self.problem: raise ValueError(f'{self.path}: {self.problem}')
                with atomic_save(self.path, 'w') as f: json.dump(data, f)

    def update(self, fn):
        "Re-read, change and save data under an exclusive file lock."
        with self._lock:
            if self.path is None:
                fn(self.data)
                return self.data
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with self.path.with_suffix('.lock').open('a') as lock:
                fcntl.flock(lock, fcntl.LOCK_EX)
                try:
                    self.reload()
                    if self.problem: raise ValueError(f'{self.path}: {self.problem}')
                    fn(self._data)
                    self.save()
                    return self._data
                finally: fcntl.flock(lock, fcntl.LOCK_UN)

`load(reg, roots=(), cfg=None, project=False, paths=(), only=None)` discovers `ramabana.mods` entry points, then configuration `extensions` and `mods`, then project `.leela/extensions` and `.leela/mods` when `project=True`, then explicit `paths`. A later source replaces an earlier source with the same name before setup runs. Names starting with `_` or `.` are skipped. A folder mod contains `mod.py`, optional `mod.toml` metadata, and optional `skills/NAME/SKILL.md` files. `reg.mods` holds records with `name`, `source`, `on`, and `note`. `reg.notes` keeps load and failure lines. `<cfg>/hooks.json` still registers shell hooks through `shell_hooks`.

`reg.mod` and `reg.home` identify the owning mod during setup and inside its callbacks, commands and tools. Tool wrappers preserve names, signatures, docstrings and annotations. The registry does not create the home folder. Explicit file mods outside `mods` or `extensions` use `<cfg>/mods/NAME` when `cfg` is set. Without `cfg`, they use `path.parent/NAME`. When `only` is a collection of names, other mods remain off with the note `not in this vesha`. Their modules do not run until `on_mod(name)` enables them. `off(name)` removes that mod's registrations and cancels its timers. `on_mod(name)` runs setup again. Unknown names raise `KeyError` with the known names. `drop_loaded()` removes the loaded mods but keeps registrations made outside the load.

`reg.status_segments` contains guarded callables taking an agent. `reg.bands` contains guarded callables taking a view, each with a `rows` limit. The frontend supplies the view fields `agent`, `busy`, `t`, `width`, and `rows`. `drain_notices()` returns dictionaries with `mod` and `text`. It clears the queue. `on_redraw` is a frontend callback and must be safe to call from worker threads. Registry hooks and timers retain their mod identity for notices and new registrations. `reg.thread(fn, *args, **kw)` starts a daemon thread with the calling mod's context. Registrations and notices from it belong to that mod. Plain `threading.Thread` loses ownership; its notices have `mod=None`. One-shot timers remove their registry entries when they finish. Failed repeating timers and cancelled timers remove their entries too.

`reg.store(default=None)` returns the calling mod's JSON store. `store.data` loads a dictionary from `<home>/state.json` on first access. A missing or unreadable file uses `default` or `{}`. `store.save()` creates the folder and writes atomically through a temporary file beside the target. `store.update(fn)` takes an exclusive lock on `<home>/state.lock`, re-reads the file, calls `fn(data)`, saves and returns the data. A mod without a home uses memory only.

`check(path)` returns a dictionary containing `tools`, `commands`, `skills`, `hooks` counts by event, `status`, `bands` row limits, `timers` intervals and repeat flags, and `mods` records. `tool_docs` adds each tool's `name`, `signature` and `doc`. It runs setup but starts no registry timers or registry threads. It is not a sandbox for arbitrary Python side effects.

A hook may return a value. `fire(event, ...)` runs the event's hooks in registration order and returns their return values, so a `before_tool` hook can return a reason to deny the call or replacement arguments, and an `after_tool` hook a rewritten result. `each(event)` returns live `(mod_name_or_None, callable)` pairs in registration order. Each callable uses the same failure rules as `fire`. Mod hooks on `before_tool` and `before_turn` fail closed with `mod NAME failed: <error>` and run again on the next event. A failing mod approval policy returns `deny`. These decision callbacks record each distinct error once. Other failing mod hooks, status segments, bands and timers are recorded once and skipped on later calls. Hooks without a mod owner record every failure and run again next time. `SystemExit` during module loading or setup disables the mod. `KeyboardInterrupt` propagates. `load` records its registration span even when interrupted. Already-running calls are not interrupted. Re-enabling a mod creates fresh registrations. A shell hook is a command listed in `hooks.json` as `{event, match, command}`, where `match` is a glob on the tool name. It receives the event as JSON on standard input and may answer with a JSON object on standard output holding `deny`, `args` or `result`. Anything else it prints is ignored.

A real extension, with a real `setup(reg)`, loaded from disk, alongside a file that fails to load and one with no `setup`:

In [ ]:
extdir = tmp/'extensions'
extdir.mkdir()
(extdir/'wc.py').write_text('''
def setup(reg):
    @reg.tool
    def word_count(path: str) -> str:
        "How many words are in a file."
        return str(len(open(path).read().split()))
    reg.command('wc', lambda agent, arg: 'counted ' + arg, help='count words')
    reg.skill('house-style', 'Dense lines. Short names.', 'How code is written here.')
''')
(extdir/'helpers.py').write_text('SHARED = 1\n')
(extdir/'boom.py').write_text('raise RuntimeError("no")\n')
reg = load(Registry(), cfg=tmp)
reg.notes

['boom.py: failed to load (RuntimeError: no)',
 'helpers.py: loaded, no setup()',
 'wc.py: 1 tool(s), 1 skill(s), 1 command(s)']

In [ ]:
test_eq([t.__name__ for t in reg.tools], ['word_count'])
test_eq(list(reg.commands), ['wc'])
test_eq([s.name for s in reg.skills], ['house-style'])
test_eq(reg.skills[0].text(), 'Dense lines. Short names.')
test_eq(reg.tools[0](str(extdir/'helpers.py')), '3')          # the tool really runs
assert any('helpers.py: loaded, no setup()' in n for n in reg.notes), reg.notes
assert any(n.startswith('boom.py: failed to load') for n in reg.notes), reg.notes

In [ ]:
(tmp/'hooks.json').write_text(json.dumps([
    {'event': 'before_tool', 'match': 'run_*',
     'command': "python3 -c 'import sys, json; d = json.load(sys.stdin); print(json.dumps({\"deny\": \"no rm\"}) if \"rm\" in d[\"args\"][2][\"command\"] else \"logged\")'"},
    {'event': 'nonsense', 'command': 'true'}]))
reg = load(Registry(), cfg=tmp)
test_eq(reg.fire('before_tool', None, 'run_shell', {'command': 'rm -rf x'}), ['no rm'])
test_eq(reg.fire('before_tool', None, 'run_shell', {'command': 'ls'}), [None])
test_eq(reg.fire('before_tool', None, 'view_file', {}), [None])
assert any(n.endswith('1 hook(s)') or 'skipped' in n for n in reg.notes), reg.notes
test_eq(reg.drop_loaded().hooks['before_tool'], [])

Reloading re-reads the extension files, and must not throw away what the running program registered since. `load` records which registrations came from the files (`mark` and `mark_loaded`), and `drop_loaded` removes exactly those and keeps the rest. `loaded` says whether the files are currently loaded.

In [ ]:
live = load(Registry(), cfg=tmp)
def live_tool(x: str) -> str: return x
live.tool(live_tool); live.command('mine', lambda a, arg: arg); live.on('after_tool', print)
live.drop_loaded()
[t.__name__ for t in live.tools], list(live.commands), len(live.hooks['after_tool']), live.loaded

In [ ]:
test_eq([t.__name__ for t in live.tools], ['live_tool'])   # the file's word_count went, mine stayed
test_eq(list(live.commands), ['mine'])
test_eq(len(live.hooks['after_tool']), 1)
test_eq([s.name for s in live.skills], [])
test_eq(live.loaded, False)
load(live, cfg=tmp)                                       # the files come back on the next read
test_eq([t.__name__ for t in live.tools], ['live_tool', 'word_count'])
live.drop_loaded()                                        # and a second round is not lossy
test_eq([t.__name__ for t in live.tools], ['live_tool'])
test_eq(Registry().drop_loaded().tools, [])               # never loaded: nothing to take back

Hooking an unknown event raises an error:

In [ ]:
test_fail(lambda: reg.on('before_lunch', print), contains='unknown event')

A failing hook is recorded in `notes` and does not end the turn. Its result is left out of what `fire` returns.

In [ ]:
fired = []
reg.on('before_turn', lambda **kw: 1/0)
reg.on('before_turn', lambda **kw: fired.append(kw))
reg.fire('before_turn', prompt='hello'), fired, reg.notes[-1]

(1,
 [{'prompt': 'hello'}],
 'before_turn hook failed: ZeroDivisionError: division by zero')

In [ ]:
test_eq(fired, [{'prompt': 'hello'}])
test_eq(reg.fire('before_turn', prompt='again'), [None])
assert reg.notes[-1].startswith('before_turn hook failed'), reg.notes[-1]

An extension may replace the approval policy, the function that decides whether a tool call goes ahead. The last one registered wins, and with none registered `approve` is None.

In [ ]:
@reg.approval
def approve_reads(tool, **kw): return not tool.startswith('write_')
reg.approve, reg.approve('view_file'), reg.approve('write_file')

(<function __main__.approve_reads(tool, **kw)>, True, False)

In [ ]:
test_eq(reg.approve.__name__, 'approve_reads')
test_eq(reg.approve('view_file'), True)
test_eq(reg.approve('write_file'), False)
test_eq(Registry().approve, None)                         # nothing registered approves everything

Extension folders inside a project load only with `project=True`. Cloning a repository never runs its code.

In [ ]:
test_eq([str(p) for p in ext_dirs(roots=['/proj'], cfg='/cfg')], ['/cfg/extensions'])
test_eq([str(p) for p in ext_dirs(roots=['/proj'], cfg='/cfg', project=True)][-1],
        '/proj/.leela/extensions')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

`session_end` hooks run when the host closes the session. Status segments and bands return frontend data. `Registry.status_texts(agent)` returns nonempty status strings in registration order. A nested call on the same thread returns no segments. A failed segment does not prevent later segments from running. Notices drain once.

Use `Registry.add_mod(name, setup, home=None)` to unit-test a mod without writing a mod file. It runs `setup(reg)` with the same ownership and callback context as a loaded mod. It returns the mod record. Pass `home` for a persistent store. Without `home`, the store stays in memory.

`store.data` returns the default when `state.json` cannot be read or parsed as a JSON object. `store.problem` holds the reason after a read and is empty when the read succeeds or the file is absent. `store.save()` and `store.update(fn)` raise `ValueError` naming the file while it has a problem. Repair or remove the file, then call `store.reload()` to re-read it and clear the problem.

In [ ]:
def setup_example(reg):
    reg.on('session_end', lambda agent: 'closed')
    reg.status(lambda agent: 'ready')
    reg.band(lambda view: ['one row'], rows=2)
    reg.notify('Saved')

mod_example = Registry()
example_mod = mod_example.add_mod('example', setup_example)
test_eq(mod_example.fire('session_end', None), ['closed'])
test_eq(mod_example.status_texts(None), ['ready'])
test_eq(mod_example.bands[0](None), ['one row'])
test_eq(mod_example.drain_notices(), [dict(mod='example', text='Saved')])
test_eq(mod_example.drain_notices(), [])
mod_example.bands[0].rows